In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv
/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv
/kaggle/input/datasets/adityachandra1611/sample-submission/sample_submission.csv


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# -----------------------------
# 1. Load data
# -----------------------------
train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv")
sample_submission = pd.read_csv("/kaggle/input/datasets/adityachandra1611/sample-submission/sample_submission.csv")

# -----------------------------
# 2. Feature engineering helper
# -----------------------------
def add_features(df):
    df = df.copy()

    # Convert timestamp to datetime if possible
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")

        # If enough valid timestamps exist, create time-based features
        if ts.notna().sum() > 0:
            df["ts_year"] = ts.dt.year
            df["ts_month"] = ts.dt.month
            df["ts_day"] = ts.dt.day
            df["ts_dayofweek"] = ts.dt.dayofweek
            df["ts_hour"] = ts.dt.hour
            df["ts_minute"] = ts.dt.minute
            df["ts_weekend"] = (ts.dt.dayofweek >= 5).astype(int)

        # Drop raw timestamp after feature extraction
        df.drop(columns=["timestamp"], inplace=True)

    # Make sure day is numeric if possible
    if "day" in df.columns:
        df["day"] = pd.to_numeric(df["day"], errors="coerce")

    # Convert object columns to string and fill missing values
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("missing").astype(str)

    return df

# -----------------------------
# 3. Prepare train/test
# -----------------------------
target = "demand"
id_col = "Index"

X = train.drop(columns=[target])
y = train[target].copy()
X_test = test.copy()

train_ids = X[id_col]
test_ids = X_test[id_col]

X = X.drop(columns=[id_col])
X_test = X_test.drop(columns=[id_col])

X = add_features(X)
X_test = add_features(X_test)

# Align columns just in case
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)

# -----------------------------
# 4. Detect categorical columns
# -----------------------------
cat_cols = [col for col in X.columns if X[col].dtype == "object"]

# Fill missing values
for col in X.columns:
    if col in cat_cols:
        X[col] = X[col].fillna("missing").astype(str)
        X_test[col] = X_test[col].fillna("missing").astype(str)
    else:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

# CatBoost needs categorical column indices
cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

# -----------------------------
# 5. K-Fold training
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    train_pool = Pool(X_train, y_train, cat_features=cat_feature_indices)
    valid_pool = Pool(X_valid, y_valid, cat_features=cat_feature_indices)
    test_pool = Pool(X_test, cat_features=cat_feature_indices)

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="R2",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        random_seed=42,
        od_type="Iter",
        od_wait=200,
        subsample=0.8,
        colsample_bylevel=0.8,
        verbose=200
    )

    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_preds[valid_idx] = model.predict(X_valid)
    test_preds += model.predict(X_test) / kf.n_splits

    fold_r2 = r2_score(y_valid, oof_preds[valid_idx])
    print(f"Fold {fold} R2: {fold_r2:.5f}")

# -----------------------------
# 6. Overall CV score
# -----------------------------
overall_r2 = r2_score(y, oof_preds)
print(f"\nOverall CV R2: {overall_r2:.5f}")

# -----------------------------
# 7. Create submission
# -----------------------------



print(len(test_preds))
print(len(test))
print(len(sample_submission))

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_preds
})

submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission.head())
submission.to_csv("/kaggle/working/submission.csv", index=False)

/tmp/ipykernel_16/1152959683.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df["timestamp"], errors="coerce")
/tmp/ipykernel_16/1152959683.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df["timestamp"], errors="coerce")


0:	learn: 0.0440162	test: 0.0448910	best: 0.0448910 (0)	total: 104ms	remaining: 8m 37s
200:	learn: 0.8924246	test: 0.8949018	best: 0.8949018 (200)	total: 6s	remaining: 2m 23s
400:	learn: 0.9088712	test: 0.9080088	best: 0.9080088 (400)	total: 13.2s	remaining: 2m 31s
600:	learn: 0.9173026	test: 0.9133876	best: 0.9133909 (599)	total: 20s	remaining: 2m 26s
800:	learn: 0.9226882	test: 0.9164021	best: 0.9164021 (800)	total: 26.8s	remaining: 2m 20s
1000:	learn: 0.9261363	test: 0.9177358	best: 0.9177358 (1000)	total: 33.8s	remaining: 2m 15s
1200:	learn: 0.9293565	test: 0.9192191	best: 0.9192205 (1197)	total: 40.8s	remaining: 2m 9s
1400:	learn: 0.9316548	test: 0.9199572	best: 0.9199572 (1400)	total: 47.6s	remaining: 2m 2s
1600:	learn: 0.9339959	test: 0.9206590	best: 0.9206618 (1597)	total: 54.3s	remaining: 1m 55s
1800:	learn: 0.9360574	test: 0.9211525	best: 0.9211807 (1786)	total: 1m 1s	remaining: 1m 48s
2000:	learn: 0.9378525	test: 0.9216860	best: 0.9216924 (1994)	total: 1m 8s	remaining: 1m 42

In [3]:
import os
print(os.path.exists("/kaggle/working/submission.csv"))

True


In [4]:
!ls -lh /kaggle/working

total 1.1M
drwxr-xr-x 5 root root 4.0K May 31 15:22 catboost_info
---------- 1 root root  39K May 31 15:36 __notebook__.ipynb
-rw-r--r-- 1 root root 1.1M May 31 15:36 submission.csv
